# Alpha101 Adaptive Research Factory

Notebook-first workflow for all 101 Kakushadze Formulaic Alphas on the cached India equity universes. The goal is not to force every alpha into the Alpha#1 repair path; it is to classify formula computability, input quality, family behavior, transform compatibility, decay, and portfolio usefulness versus the correct active equal-weight benchmark.

Primary formula source: Kakushadze, *101 Formulaic Alphas*, arXiv:1601.00991.

## Run Controls

`RUN_ALPHA101_REFRESH=False` is the normal mode. It loads existing artifacts if present, otherwise it resumes from per-alpha task caches and computes only what is missing. Set it to `True` only when you intentionally want to rebuild all task caches.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

from research.alpha101_engine import ALPHA101_ARTIFACT_DIR, load_panel
from research.alpha101_factory import run_alpha101_factory
from research.alpha101_robustness import run_alpha101_robustness, run_alpha101_robustness_batch2

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RUN_ALPHA101_REFRESH = False
ALPHA101_REAGGREGATE = False
RUN_ALPHA101_ROBUSTNESS_REFRESH = False
RUN_ALPHA101_ROBUSTNESS_BATCH2_REFRESH = False
ALPHA101_MAX_WORKERS = 4
ARTIFACT_DIR = ALPHA101_ARTIFACT_DIR
ARTIFACT_DIR

PosixPath('research/artifacts/alpha101_research_factory')

## Data And Universe Audit

The factory uses cached real OHLCV data only. The close used by formulas is adjusted close; open/high/low are adjusted by the same close adjustment factor so mixed OHLC formulas are not distorted by corporate actions. VWAP is a labeled proxy: `(adjusted_high + adjusted_low + adjusted_close) / 3`.

In [2]:
panel_rows = []
for panel_name in ["nifty500", "expanded"]:
    panel = load_panel(panel_name)
    panel_rows.append({
        "panel": panel_name,
        "start": panel.close.index.min().date(),
        "end": panel.close.index.max().date(),
        "sessions": len(panel.close),
        "symbols": panel.close.shape[1],
        "median_active_names": panel.active_mask.sum(axis=1).median(),
        "median_high_vol_names": panel.high_vol_mask.sum(axis=1).median(),
        "pit_risk": panel.pit_risk,
    })
display(pd.DataFrame(panel_rows))

,panel,start,end,sessions,symbols,median_active_names,median_high_vol_names,pit_risk
0,nifty500,2018-01-01,2026-05-19,2069,503,401.0,100.0,current_snapshot_constituents_no_point_in_time...
1,expanded,2018-01-01,2026-05-19,2069,751,568.0,100.0,current_snapshot_constituents_no_point_in_time...


## Run Or Load Factory

This single call produces every artifact: registry, input-quality report, operator validation, formula validation, IC panels, transform grid, decay report, portfolio report, leaderboard, shortlist, and final markdown report.

In [3]:
outputs = run_alpha101_factory(
    max_workers=ALPHA101_MAX_WORKERS,
    refresh=RUN_ALPHA101_REFRESH,
    progress=True,
    reaggregate=ALPHA101_REAGGREGATE,
)
{k: v.shape for k, v in outputs.items()}

{'registry': (101, 8),
 'input_quality': (101, 4),
 'operator_validation': (7, 2),
 'formula_validation': (202, 8),
 'family_classification': (101, 2),
 'transform_compatibility': (6, 4),
 'metric_panel': (8992, 11),
 'transform_grid': (4000, 10),
 'decay_report': (800, 7),
 'portfolio_report': (81280, 35),
 'leaderboard': (201, 18),
 'shortlist': (50, 18)}

## Formula Registry Audit

In [4]:
registry = outputs["registry"]
display(registry.head(20))
display(registry["input_quality_tier"].value_counts(dropna=False).rename("alphas").to_frame())
display(registry["family"].value_counts(dropna=False).rename("alphas").to_frame())

,alpha_id,family,required_inputs,input_quality_tier,delay_tag,formula_text,callable_name,notes
0,alpha001,price_reversal,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#001 from Kakushadze 101 Formulaic Alphas...,alpha001,NaN
1,alpha002,volume_liquidity,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#002 from Kakushadze 101 Formulaic Alphas...,alpha002,NaN
2,alpha003,volume_liquidity,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#003 from Kakushadze 101 Formulaic Alphas...,alpha003,NaN
3,alpha004,price_reversal,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#004 from Kakushadze 101 Formulaic Alphas...,alpha004,NaN
4,alpha005,volume_liquidity,"open,high,low,close,volume,vwap",proxy_vwap,paper_delay_mixed,Alpha#005 from Kakushadze 101 Formulaic Alphas...,alpha005,NaN
5,alpha006,volume_liquidity,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#006 from Kakushadze 101 Formulaic Alphas...,alpha006,NaN
6,alpha007,price_reversal,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#007 from Kakushadze 101 Formulaic Alphas...,alpha007,NaN
7,alpha008,momentum_trend,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#008 from Kakushadze 101 Formulaic Alphas...,alpha008,NaN
8,alpha009,price_reversal,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#009 from Kakushadze 101 Formulaic Alphas...,alpha009,NaN
9,alpha010,price_reversal,"open,high,low,close,volume",exact_ohlcv,paper_delay_mixed,Alpha#010 from Kakushadze 101 Formulaic Alphas...,alpha010,NaN


,alphas
input_quality_tier,
exact_ohlcv,51
proxy_vwap,31
proxy_vwap+snapshot_industry,14
snapshot_industry,4
missing_cap,1


,alphas
family,
volume_liquidity,40
price_reversal,26
industry_neutral_cross_section,18
momentum_trend,14
volatility_range,2
hybrid_or_unknown,1


## Input Quality Audit

`exact_ohlcv` alphas use cached adjusted OHLCV directly. `proxy_vwap` alphas use typical price as VWAP proxy. `snapshot_industry` alphas use current constituent industry metadata and are explicitly not PIT industry backtests. `missing_cap` is untestable with the current cache.

In [5]:
input_quality = outputs["input_quality"]
display(input_quality.groupby("input_quality_tier", dropna=False).size().rename("alphas").reset_index())
display(input_quality[input_quality["input_quality_tier"].str.contains("missing", na=False)])

,input_quality_tier,alphas
0,exact_ohlcv,51
1,missing_cap,1
2,proxy_vwap,31
3,proxy_vwap+snapshot_industry,14
4,snapshot_industry,4


,alpha_id,required_inputs,input_quality_tier,notes
55,alpha056,"open,high,low,close,volume,cap",missing_cap,market cap is unavailable in current cache


## Operator Validation

In [6]:
operator_validation = outputs["operator_validation"]
display(operator_validation)
assert operator_validation["passed"].all(), "Operator validation failed"

,check,passed
0,rank_bounds,True
1,delta,True
2,delay,True
3,ts_argmax,True
4,ts_argmin,True
5,signed_power,True
6,scale_abs_sum,True


## Formula Validation

Every formula must either compute on each panel or be explicitly marked untestable/failed. This is where formula, missing input, and numerical failures are separated before judging signal quality.

In [7]:
formula_validation = outputs["formula_validation"]
display(formula_validation.groupby(["panel", "computed"], dropna=False).size().rename("count").reset_index())
display(formula_validation[~formula_validation["computed"].fillna(False)].head(50))
display(formula_validation.sort_values("non_null_scores", ascending=False).head(20))

,panel,computed,count
0,expanded,False,1
1,expanded,True,100
2,nifty500,False,1
3,nifty500,True,100


,panel,alpha_id,computed,reason,non_null_scores,median_daily_coverage,min_score,max_score
110,nifty500,alpha056,False,market cap is unavailable in current cache,0,NaN,NaN,NaN
111,expanded,alpha056,False,market cap is unavailable in current cache,0,NaN,NaN,NaN


,panel,alpha_id,computed,reason,non_null_scores,median_daily_coverage,min_score,max_score
201,expanded,alpha101,True,NaN,1194887,568.0,-0.999998,9.999982e-01
41,expanded,alpha021,True,NaN,1194887,568.0,-1.000000,1.000000e+00
55,expanded,alpha028,True,NaN,1194887,568.0,-0.445772,5.398708e-01
65,expanded,alpha033,True,NaN,1194887,568.0,0.001340,1.000000e+00
81,expanded,alpha041,True,NaN,1194887,568.0,-1891.193142,2.177072e+03
83,expanded,alpha042,True,NaN,1194887,568.0,0.001340,7.030000e+02
87,expanded,alpha044,True,NaN,1194887,568.0,-1.000000,1.221770e+00
107,expanded,alpha054,True,NaN,1194887,568.0,-13.912710,8.881784e-12
109,expanded,alpha055,True,NaN,1194887,568.0,-1.000000,9.999006e-01
121,expanded,alpha061,True,NaN,1194887,568.0,0.000000,1.000000e+00


## Family And Transform Compatibility

Transform selection is reported separately from baseline metrics to reduce data-mining fog. Family rules determine what gets tried; they do not declare a transform valid by itself.

In [8]:
display(outputs["family_classification"].head(30))
display(outputs["transform_compatibility"])

,alpha_id,family
0,alpha001,price_reversal
1,alpha002,volume_liquidity
2,alpha003,volume_liquidity
3,alpha004,price_reversal
4,alpha005,volume_liquidity
5,alpha006,volume_liquidity
6,alpha007,price_reversal
7,alpha008,momentum_trend
8,alpha009,price_reversal
9,alpha010,price_reversal


,family,signal_transforms,portfolio_templates,note
0,hybrid_or_unknown,"raw,rank_centered,zscore,winsor_zscore,rank_no...","top10,long_short_10,score_tilt,overlay20,ewm3_...","Transforms are family-compatible candidates, n..."
1,industry_neutral_cross_section,"raw,rank_centered,zscore,winsor_zscore,rank_no...","long_short_10,score_tilt,overlay20,ewm3_overlay20","Transforms are family-compatible candidates, n..."
2,momentum_trend,"raw,rank_centered,zscore,winsor_zscore,rank_no...","top10,score_tilt,overlay20,ewm5_overlay20,ewm1...","Transforms are family-compatible candidates, n..."
3,price_reversal,"raw,rank_centered,zscore,winsor_zscore,rank_no...","top10,long_short_10,score_tilt,overlay20,ewm3_...","Transforms are family-compatible candidates, n..."
4,volatility_range,"raw,rank_centered,zscore,winsor_zscore,rank_no...","top10,long_short_10,score_tilt,overlay20,ewm3_...","Transforms are family-compatible candidates, n..."
5,volume_liquidity,"raw,rank_centered,zscore,winsor_zscore,rank_no...","top10,long_short_10,score_tilt,overlay20,ewm3_...","Transforms are family-compatible candidates, n..."


## IC And Horizon Diagnostics

This table ranks formula/transform/horizon combinations by mean rank IC. Use it to answer: is the formula directionally informative before portfolio construction?

In [9]:
metric_panel = outputs["metric_panel"]
ic_cols = ["panel", "alpha_id", "family", "transform", "horizon_days", "mean_rank_ic", "rank_icir", "positive_ic_rate", "observations"]
display(metric_panel.sort_values("mean_rank_ic", ascending=False)[ic_cols].head(40))
display(metric_panel.sort_values("mean_rank_ic", ascending=True)[ic_cols].head(20))

,panel,alpha_id,family,transform,horizon_days,mean_rank_ic,rank_icir,positive_ic_rate,observations
8606,expanded,alpha097,industry_neutral_cross_section,zscore,5,0.209274,0.550763,0.608696,23
8610,expanded,alpha097,industry_neutral_cross_section,winsor_zscore,5,0.209274,0.550763,0.608696,23
8598,expanded,alpha097,industry_neutral_cross_section,raw,5,0.209274,0.550763,0.608696,23
8614,expanded,alpha097,industry_neutral_cross_section,rank_normal,5,0.209274,0.550763,0.608696,23
8618,expanded,alpha097,industry_neutral_cross_section,clipped_zscore,5,0.209274,0.550763,0.608696,23
8602,expanded,alpha097,industry_neutral_cross_section,rank_centered,5,0.209274,0.550763,0.608696,23
8622,expanded,alpha097,industry_neutral_cross_section,tanh_z,5,0.209274,0.550763,0.608696,23
8625,expanded,alpha097,industry_neutral_cross_section,industry_residual,3,0.193357,0.611293,0.652174,23
8626,expanded,alpha097,industry_neutral_cross_section,industry_residual,5,0.178637,0.674246,0.782609,23
8627,expanded,alpha097,industry_neutral_cross_section,industry_residual,10,0.173782,0.656117,0.782609,23


,panel,alpha_id,family,transform,horizon_days,mean_rank_ic,rank_icir,positive_ic_rate,observations
7442,nifty500,alpha084,price_reversal,threshold_top_bottom,5,-0.043135,-0.232268,0.404244,1932
7441,nifty500,alpha084,price_reversal,threshold_top_bottom,3,-0.040402,-0.220800,0.407446,1934
3173,expanded,alpha035,momentum_trend,winsor_zscore,3,-0.040295,-0.307667,0.380558,1934
3185,expanded,alpha035,momentum_trend,tanh_z,3,-0.040287,-0.307603,0.381075,1934
3165,expanded,alpha035,momentum_trend,rank_centered,3,-0.040287,-0.307603,0.381075,1934
3177,expanded,alpha035,momentum_trend,rank_normal,3,-0.040287,-0.307603,0.381075,1934
3205,expanded,alpha035,momentum_trend,signed_square,3,-0.040287,-0.307603,0.381075,1934
3169,expanded,alpha035,momentum_trend,zscore,3,-0.040287,-0.307603,0.381075,1934
3161,expanded,alpha035,momentum_trend,raw,3,-0.040287,-0.307603,0.381075,1934
3181,expanded,alpha035,momentum_trend,clipped_zscore,3,-0.040287,-0.307601,0.381075,1934


## Decay Diagnostics

The decay report compares IC by era and includes a `recent_minus_old` row. This is not a final verdict by itself, but it flags families that appear to have weakened in the latest period.

In [10]:
decay_report = outputs["decay_report"]
display(decay_report[decay_report["era"].eq("recent_minus_old")].sort_values("mean_rank_ic").head(30))
display(decay_report.pivot_table(index="family", columns="era", values="mean_rank_ic", aggfunc="median"))

,panel,alpha_id,family,era,mean_rank_ic,positive_ic_rate,observations
543,expanded,alpha069,industry_neutral_cross_section,recent_minus_old,-0.031093,NaN,1802
515,nifty500,alpha066,price_reversal,recent_minus_old,-0.027950,NaN,1763
151,expanded,alpha019,price_reversal,recent_minus_old,-0.024280,NaN,1681
527,expanded,alpha067,industry_neutral_cross_section,recent_minus_old,-0.022155,NaN,1802
411,nifty500,alpha052,price_reversal,recent_minus_old,-0.022103,NaN,1691
647,expanded,alpha082,industry_neutral_cross_section,recent_minus_old,-0.020281,NaN,1802
519,expanded,alpha066,price_reversal,recent_minus_old,-0.020102,NaN,1763
783,expanded,alpha099,volume_liquidity,recent_minus_old,-0.020098,NaN,1802
663,expanded,alpha084,price_reversal,recent_minus_old,-0.020050,NaN,1802
183,expanded,alpha023,price_reversal,recent_minus_old,-0.019785,NaN,1800


era,2018_2020,2021_2023,2024_2026,recent_minus_old
family,,,,
industry_neutral_cross_section,0.013416,0.014517,0.007347,-0.004515
momentum_trend,0.014312,0.015415,0.012648,0.001234
price_reversal,0.017678,0.017618,0.015527,-0.004053
volatility_range,0.001538,0.007003,0.003245,-0.000365
volume_liquidity,0.015999,0.016497,0.014010,-0.000419


## Portfolio Reality Check

The primary portfolio metric is active return versus the equal-weight active universe after costs. The report keeps alpha return, benchmark return, excess return, turnover, drawdown, Sharpe, Sortino, and hit rate side by side.

In [11]:
portfolio_report = outputs["portfolio_report"]
portfolio_cols = [
    "panel", "alpha_id", "family", "signal_transform", "mask", "strategy", "cost_bps",
    "alpha_cagr", "benchmark_cagr", "active_cagr", "active_sharpe", "alpha_avg_daily_turnover", "active_max_drawdown", "avg_names",
]
display(portfolio_report.query("cost_bps == 20.0").sort_values("active_sharpe", ascending=False)[portfolio_cols].head(50))
display(portfolio_report.query("cost_bps == 20.0").pivot_table(index="family", columns="strategy", values="active_sharpe", aggfunc="median"))

,panel,alpha_id,family,signal_transform,mask,strategy,cost_bps,alpha_cagr,benchmark_cagr,active_cagr,active_sharpe,alpha_avg_daily_turnover,active_max_drawdown,avg_names
19245,expanded,alpha023,price_reversal,winsor_zscore,high_vol_top100,ewm5_overlay20,20.0,0.248569,0.208596,0.032707,1.830745,0.067603,-0.019969,336.036082
20181,expanded,alpha024,price_reversal,rank_centered,high_vol_top100,ewm5_overlay20,20.0,0.247707,0.208596,0.032333,1.830516,0.062707,-0.021740,336.180928
20177,expanded,alpha024,price_reversal,rank_centered,high_vol_top100,ewm3_overlay20,20.0,0.247242,0.208596,0.031937,1.828006,0.063269,-0.020729,336.194330
20221,expanded,alpha024,price_reversal,ewm3,high_vol_top100,overlay20,20.0,0.247187,0.208596,0.031887,1.827184,0.063243,-0.020729,336.192268
20173,expanded,alpha024,price_reversal,rank_centered,high_vol_top100,overlay20,20.0,0.246560,0.208596,0.031306,1.827057,0.063877,-0.019380,336.151546
19285,expanded,alpha023,price_reversal,threshold_top_bottom,high_vol_top100,overlay20,20.0,0.248413,0.208596,0.032316,1.817007,0.071410,-0.021113,336.380412
20225,expanded,alpha024,price_reversal,ewm3,high_vol_top100,ewm3_overlay20,20.0,0.247073,0.208596,0.031814,1.802853,0.062786,-0.021930,336.193299
19241,expanded,alpha023,price_reversal,winsor_zscore,high_vol_top100,ewm3_overlay20,20.0,0.247482,0.208596,0.031773,1.797509,0.068344,-0.019848,336.211340
20229,expanded,alpha024,price_reversal,ewm3,high_vol_top100,ewm5_overlay20,20.0,0.247157,0.208596,0.031885,1.793261,0.062316,-0.022484,336.181959
20205,expanded,alpha024,price_reversal,winsor_zscore,high_vol_top100,ewm5_overlay20,20.0,0.247675,0.208596,0.032512,1.775010,0.062414,-0.017198,333.472165


strategy,ewm10_overlay20,ewm3_overlay20,ewm5_overlay20,long_short_10,overlay20,score_tilt,top10
family,,,,,,,
industry_neutral_cross_section,NaN,-0.329962,NaN,-1.473058,-0.723667,-0.119580,NaN
momentum_trend,-0.424741,NaN,-0.436761,NaN,-0.953551,-0.104303,-0.600746
price_reversal,NaN,-0.250092,-0.217616,-1.523267,-0.779972,-0.194507,-0.736027
volatility_range,NaN,-0.464305,NaN,-1.856700,-1.161949,-0.225757,-1.297822
volume_liquidity,NaN,-0.484350,NaN,-1.453183,-0.837015,-0.107578,-0.647205


## Transform Grid Summary

This aggregates transform/portfolio templates across alphas. Treat it as a research-map of where signal extraction tends to survive costs, not as a tradable model selection step.

In [12]:
transform_grid = outputs["transform_grid"]
display(transform_grid.sort_values("mean_active_sharpe", ascending=False).head(60))

,panel,family,signal_transform,mask,strategy,cost_bps,mean_active_sharpe,median_active_sharpe,positive_active_rate,alphas
1983,expanded,volume_liquidity,ewm3,high_vol_top100,ewm3_overlay20,50.0,1.258830,1.235859,1.0,40
243,expanded,industry_neutral_cross_section,style_residual,high_vol_top100,ewm3_overlay20,50.0,1.256822,1.208963,1.0,18
1991,expanded,volume_liquidity,ewm3,high_vol_top100,overlay20,50.0,1.250953,1.251061,1.0,40
2223,expanded,volume_liquidity,rank_centered,high_vol_top100,ewm3_overlay20,50.0,1.249373,1.249047,1.0,40
2343,expanded,volume_liquidity,winsor_zscore,high_vol_top100,ewm3_overlay20,50.0,1.242250,1.240807,1.0,40
1375,expanded,price_reversal,winsor_zscore,high_vol_top100,ewm5_overlay20,50.0,1.237148,1.165008,1.0,26
943,expanded,price_reversal,ewm3,high_vol_top100,ewm5_overlay20,50.0,1.231836,1.217817,1.0,26
1227,expanded,price_reversal,threshold_top_bottom,high_vol_top100,ewm3_overlay20,50.0,1.231060,1.223224,1.0,26
1087,expanded,price_reversal,rank_centered,high_vol_top100,ewm5_overlay20,50.0,1.227793,1.213482,1.0,26
147,expanded,industry_neutral_cross_section,rank_centered,high_vol_top100,ewm3_overlay20,50.0,1.220652,1.201026,1.0,18


## Leaderboard And Candidate Shortlist

The leaderboard combines IC, portfolio active Sharpe after 20 bps, turnover, and recent decay. The shortlist is deliberately research-oriented: candidates and feature-only alphas are kept separate from tradable approval.

In [13]:
leaderboard = outputs["leaderboard"]
shortlist = outputs["shortlist"]
leader_cols = [
    "panel", "alpha_id", "family", "input_quality_tier", "classification", "best_5d_ic", "rank_icir",
    "positive_ic_rate", "best_20bps_active_sharpe", "best_signal_transform", "best_strategy", "best_mask", "recent_minus_old_ic", "research_score",
]
display(leaderboard[leader_cols].head(60))
display(shortlist[leader_cols].head(60))
display(leaderboard.groupby(["panel", "classification"], dropna=False).size().rename("count").reset_index())

,panel,alpha_id,family,input_quality_tier,classification,best_5d_ic,rank_icir,positive_ic_rate,best_20bps_active_sharpe,best_signal_transform,best_strategy,best_mask,recent_minus_old_ic,research_score
0,expanded,alpha097,industry_neutral_cross_section,proxy_vwap+snapshot_industry,candidate,0.209274,0.550763,0.608696,1.292440,rank_centered,ewm3_overlay20,high_vol_top100,NaN,2.476420
1,expanded,alpha024,price_reversal,exact_ohlcv,candidate,0.049329,0.283500,0.606108,1.830516,rank_centered,ewm5_overlay20,high_vol_top100,-0.019363,2.213010
2,expanded,alpha023,price_reversal,exact_ohlcv,candidate,0.043355,0.228532,0.586462,1.830745,winsor_zscore,ewm5_overlay20,high_vol_top100,-0.019785,2.177236
3,expanded,alpha063,industry_neutral_cross_section,proxy_vwap+snapshot_industry,candidate,0.045020,0.246328,0.599341,1.676850,rank_centered,overlay20,high_vol_top100,-0.007390,2.035333
4,expanded,alpha034,price_reversal,exact_ohlcv,candidate,0.036315,0.310307,0.620725,1.683247,ewm3,ewm3_overlay20,high_vol_top100,-0.011559,2.003310
5,expanded,alpha044,volume_liquidity,exact_ohlcv,candidate,0.056556,0.334007,0.628882,1.577352,ewm3,overlay20,high_vol_top100,0.003985,2.000538
6,expanded,alpha016,volume_liquidity,exact_ohlcv,candidate,0.059026,0.342029,0.629400,1.538856,ewm3,overlay20,high_vol_top100,-0.008524,1.974714
7,nifty500,alpha024,price_reversal,exact_ohlcv,candidate,0.042959,0.241432,0.609213,1.558933,winsor_zscore,ewm3_overlay20,seasoned_2y,-0.014536,1.925000
8,expanded,alpha040,volume_liquidity,exact_ohlcv,candidate,0.040769,0.235604,0.592133,1.583510,winsor_zscore,overlay20,high_vol_top100,0.003526,1.918782
9,expanded,alpha049,price_reversal,exact_ohlcv,candidate,0.030063,0.152101,0.552535,1.553261,winsor_zscore,ewm5_overlay20,high_vol_top100,-0.004302,1.824687


,panel,alpha_id,family,input_quality_tier,classification,best_5d_ic,rank_icir,positive_ic_rate,best_20bps_active_sharpe,best_signal_transform,best_strategy,best_mask,recent_minus_old_ic,research_score
0,expanded,alpha097,industry_neutral_cross_section,proxy_vwap+snapshot_industry,candidate,0.209274,0.550763,0.608696,1.292440,rank_centered,ewm3_overlay20,high_vol_top100,NaN,2.476420
1,expanded,alpha024,price_reversal,exact_ohlcv,candidate,0.049329,0.283500,0.606108,1.830516,rank_centered,ewm5_overlay20,high_vol_top100,-0.019363,2.213010
2,expanded,alpha023,price_reversal,exact_ohlcv,candidate,0.043355,0.228532,0.586462,1.830745,winsor_zscore,ewm5_overlay20,high_vol_top100,-0.019785,2.177236
3,expanded,alpha063,industry_neutral_cross_section,proxy_vwap+snapshot_industry,candidate,0.045020,0.246328,0.599341,1.676850,rank_centered,overlay20,high_vol_top100,-0.007390,2.035333
4,expanded,alpha034,price_reversal,exact_ohlcv,candidate,0.036315,0.310307,0.620725,1.683247,ewm3,ewm3_overlay20,high_vol_top100,-0.011559,2.003310
5,expanded,alpha044,volume_liquidity,exact_ohlcv,candidate,0.056556,0.334007,0.628882,1.577352,ewm3,overlay20,high_vol_top100,0.003985,2.000538
6,expanded,alpha016,volume_liquidity,exact_ohlcv,candidate,0.059026,0.342029,0.629400,1.538856,ewm3,overlay20,high_vol_top100,-0.008524,1.974714
7,nifty500,alpha024,price_reversal,exact_ohlcv,candidate,0.042959,0.241432,0.609213,1.558933,winsor_zscore,ewm3_overlay20,seasoned_2y,-0.014536,1.925000
8,expanded,alpha040,volume_liquidity,exact_ohlcv,candidate,0.040769,0.235604,0.592133,1.583510,winsor_zscore,overlay20,high_vol_top100,0.003526,1.918782
9,expanded,alpha049,price_reversal,exact_ohlcv,candidate,0.030063,0.152101,0.552535,1.553261,winsor_zscore,ewm5_overlay20,high_vol_top100,-0.004302,1.824687


,panel,classification,count
0,expanded,candidate,48
1,expanded,decayed,4
2,expanded,discard,48
3,nifty500,candidate,36
4,nifty500,discard,45
5,nifty500,feature_only,19
6,NaN,untestable,1


## Candidate Robustness And Data-Risk Triage

The factory leaderboard is discovery only. This stage reruns only the top candidates, selects transform/portfolio/mask combinations on train windows, scores test windows versus the equal-weight active universe, and separates exact-OHLCV, proxy-VWAP, snapshot-industry, and Alpha#1 baseline lanes.

In [14]:
robustness = run_alpha101_robustness(
    refresh=RUN_ALPHA101_ROBUSTNESS_REFRESH,
    clean_n=12,
    proxy_n=8,
    snapshot_n=8,
    progress=True,
)
{k: v.shape for k, v in robustness.items()}

{'candidate_lanes': (30, 20),
 'walk_forward': (120, 30),
 'cost_sensitivity': (480, 30),
 'universe_sensitivity': (480, 30),
 'proxy_sensitivity': (56, 14),
 'industry_snapshot_risk': (8, 13),
 'validation': (6, 3),
 'shortlist': (30, 20)}

## Robustness Validation

In [15]:
display(robustness["validation"])
assert robustness["validation"]["passed"].all(), "Robustness validation failed"

,check,passed,detail
0,alpha001_included_as_baseline,True,Alpha#1 included in baseline_alpha001 lane.
1,exact_candidates_separated,True,Exact OHLCV candidates have their own lane.
2,train_selected_only,True,Walk-forward rows select transforms/portfolios...
3,cost_grid_present,True,Cost sensitivity is generated at 10/20/35/50 bps.
4,no_full_sample_orientation,True,"Tradable signals use causal_orient, which shif..."
5,final_status_available,True,Each robustness candidate receives a final tri...


## Robustness Shortlist

`promote_to_deeper_research` survived walk-forward active-return checks after train-only selection. `feature_only` retained signal information but did not clear the portfolio hurdle. `proxy_dependent` is rejected until better VWAP data exists. `snapshot_metadata_risk` is research-only until point-in-time industry metadata exists.

In [16]:
robust_cols = [
    "panel", "alpha_id", "robustness_lane", "input_quality_tier", "final_status",
    "median_test_active_sharpe", "median_test_active_cagr", "positive_test_sharpe_rate",
    "median_test_rank_ic", "median_turnover", "worst_test_drawdown",
    "best_mask", "best_signal_transform", "best_strategy",
]
display(robustness["shortlist"].sort_values("median_test_active_sharpe", ascending=False)[robust_cols].head(60))
display(robustness["shortlist"].groupby(["robustness_lane", "final_status"], dropna=False).size().rename("count").reset_index())

,panel,alpha_id,robustness_lane,input_quality_tier,final_status,median_test_active_sharpe,median_test_active_cagr,positive_test_sharpe_rate,median_test_rank_ic,median_turnover,worst_test_drawdown,best_mask,best_signal_transform,best_strategy
22,expanded,alpha063,snapshot_metadata_risk,proxy_vwap+snapshot_industry,snapshot_metadata_risk,1.744368,0.028139,0.75,0.031102,0.065813,-0.024411,high_vol_top100,rank_centered,overlay20
3,expanded,alpha040,clean_exact_ohlcv,exact_ohlcv,promote_to_deeper_research,1.669713,0.028283,1.00,0.034546,0.066179,-0.017784,high_vol_top100,winsor_zscore,overlay20
4,expanded,alpha012,clean_exact_ohlcv,exact_ohlcv,promote_to_deeper_research,1.448578,0.026729,1.00,0.020490,0.066969,-0.019509,high_vol_top100,ewm3,ewm3_overlay20
5,expanded,alpha024,clean_exact_ohlcv,exact_ohlcv,promote_to_deeper_research,1.441989,0.027489,1.00,0.039320,0.062937,-0.016986,high_vol_top100,rank_centered,ewm5_overlay20
6,expanded,alpha044,clean_exact_ohlcv,exact_ohlcv,promote_to_deeper_research,1.364653,0.022774,1.00,0.041260,0.067882,-0.023468,high_vol_top100,ewm3,overlay20
23,expanded,alpha069,snapshot_metadata_risk,proxy_vwap+snapshot_industry,snapshot_metadata_risk,1.349114,0.022692,1.00,0.008951,0.066677,-0.016153,high_vol_top100,style_residual,overlay20
7,expanded,alpha018,clean_exact_ohlcv,exact_ohlcv,promote_to_deeper_research,1.341600,0.024729,1.00,0.022731,0.065898,-0.022355,high_vol_top100,ewm3,ewm3_overlay20
8,expanded,alpha023,clean_exact_ohlcv,exact_ohlcv,promote_to_deeper_research,1.268938,0.024179,1.00,0.029528,0.067603,-0.021113,high_vol_top100,winsor_zscore,ewm5_overlay20
9,expanded,alpha051,clean_exact_ohlcv,exact_ohlcv,promote_to_deeper_research,1.213014,0.023337,1.00,0.016318,0.068092,-0.023548,high_vol_top100,winsor_zscore,ewm5_overlay20
10,expanded,alpha049,clean_exact_ohlcv,exact_ohlcv,promote_to_deeper_research,1.205458,0.023193,0.75,0.017777,0.068093,-0.030831,high_vol_top100,winsor_zscore,ewm5_overlay20


,robustness_lane,final_status,count
0,baseline_alpha001,baseline_comparator,2
1,clean_exact_ohlcv,feature_only,1
2,clean_exact_ohlcv,promote_to_deeper_research,11
3,proxy_vwap,promote_to_deeper_research,5
4,proxy_vwap,proxy_dependent,3
5,snapshot_metadata_risk,snapshot_metadata_risk,8


## Cost And Universe Sensitivity

In [17]:
display(robustness["cost_sensitivity"].pivot_table(
    index=["panel", "alpha_id", "robustness_lane"],
    columns="cost_bps",
    values="test_active_sharpe",
    aggfunc="median",
).sort_values(20.0, ascending=False).head(40))

display(robustness["universe_sensitivity"].pivot_table(
    index=["panel", "alpha_id", "robustness_lane"],
    columns="selected_mask",
    values="test_active_sharpe",
    aggfunc="median",
).sort_values("high_vol_top100", ascending=False).head(40))

cost_bps                                      10.0      20.0      35.0      50.0
panel    alpha_id robustness_lane                                               
expanded alpha063 snapshot_metadata_risk  1.688508  1.744368  1.824200  1.899026
         alpha040 clean_exact_ohlcv       1.617580  1.669713  1.744634  1.815407
         alpha012 clean_exact_ohlcv       1.410466  1.448578  1.503291  1.554908
         alpha024 clean_exact_ohlcv       1.354246  1.441989  1.570591  1.695063
         alpha044 clean_exact_ohlcv       1.330809  1.364653  1.412338  1.440732
         alpha069 snapshot_metadata_risk  1.298146  1.349114  1.409135  1.457539
         alpha018 clean_exact_ohlcv       1.304363  1.341600  1.395425  1.446625
         alpha023 clean_exact_ohlcv       1.223509  1.268938  1.333413  1.393195
         alpha051 clean_exact_ohlcv       1.173095  1.213014  1.269788  1.321669
         alpha049 clean_exact_ohlcv       1.165527  1.205458  1.262255  1.310624
         alpha067 snapshot_metadata_risk  1.152846  1.196731  1.260091  1.320251
         alpha016 clean_exact_ohlcv       1.138477  1.183597  1.247885  1.307655
         alpha034 clean_exact_ohlcv       1.025366  1.087923  1.178648  1.265214
         alpha094 proxy_vwap              1.011040  1.066946  1.148035  1.226781
         alpha025 proxy_vwap              0.993007  1.048234  1.128878  1.206624
         alpha022 clean_exact_ohlcv       0.994671  1.004016  1.016019  1.025629
         alpha070 snapshot_metadata_risk  0.900066  0.932373  0.979114  1.023639
         alpha050 proxy_vwap              0.876142  0.901224  0.936087  0.967388
         alpha082 snapshot_metadata_risk  0.835291  0.889333  0.967697  1.042422
         alpha097 snapshot_metadata_risk  0.673825  0.880402  1.182635  1.473519
         alpha027 proxy_vwap              0.691166  0.746627  0.827525  0.905356
         alpha005 proxy_vwap              0.690741  0.722309  0.768224  0.812283
         alpha001 baseline_alpha001       0.646992  0.693653  0.761988  0.828069
         alpha036 proxy_vwap              0.509458  0.603941  0.743120  0.878570
         alpha057 proxy_vwap              0.463234  0.515791  0.593103  0.668293
         alpha089 snapshot_metadata_risk  0.493374  0.514243  0.544297  0.572735
         alpha090 snapshot_metadata_risk  0.341333  0.386491  0.452159  0.514800
         alpha041 proxy_vwap              0.296219  0.340569  0.420651  0.511563
nifty500 alpha001 baseline_alpha001       0.156818  0.170666  0.190149  0.207928
         alpha024 clean_exact_ohlcv       0.370635  0.158085 -0.327368 -0.598721

selected_mask                             all_eligible  high_vol_top100  seasoned_2y  strict_liquidity_100m
panel    alpha_id robustness_lane                                                                          
expanded alpha063 snapshot_metadata_risk     -1.489904         1.744368    -1.220563              -1.446748
         alpha040 clean_exact_ohlcv           0.367740         1.669713     0.775168              -0.832808
         alpha012 clean_exact_ohlcv          -1.931323         1.448578    -0.433002              -3.162955
         alpha024 clean_exact_ohlcv           1.835390         1.441989     1.448763               0.734334
         alpha044 clean_exact_ohlcv          -1.576713         1.364653    -2.072924              -1.556196
         alpha069 snapshot_metadata_risk     -1.933476         1.349114     0.396064              -1.493655
         alpha018 clean_exact_ohlcv          -0.284077         1.341600     1.139807              -1.445668
         alpha023 clean_exact_ohlcv           0.242870         1.268938     0.259959              -1.651030
         alpha051 clean_exact_ohlcv          -1.141842         1.213014    -0.169535              -2.588595
         alpha049 clean_exact_ohlcv          -1.125946         1.205458    -0.171524              -2.554514
         alpha067 snapshot_metadata_risk     -4.274950         1.196731    -0.926346              -3.603684
         alpha016 clean_exact_ohlcv          -1.075348         1.183597    -1.600219              -2.253476
         alpha034 clean_exact_ohlcv          -2.873445         1.087923    -2.814828              -2.815152
         alpha094 proxy_vwap                 -0.953603         1.066946    -0.402620              -1.049046
         alpha025 proxy_vwap                 -1.846597         1.048234    -1.612436              -1.646439
         alpha022 clean_exact_ohlcv          -4.520528         1.004016    -4.255439              -3.966136
         alpha070 snapshot_metadata_risk     -2.469829         0.932373    -0.337419              -1.949702
         alpha050 proxy_vwap                 -2.978573         0.901224    -2.799499              -2.215355
         alpha082 snapshot_metadata_risk     -3.243880         0.889333    -2.197446              -1.721276
         alpha097 snapshot_metadata_risk     -0.654776         0.880402    -1.036246              -0.582587
nifty500 alpha024 clean_exact_ohlcv           0.824835         0.752795     1.009739               0.668039
expanded alpha027 proxy_vwap                 -3.345029         0.746627    -3.107391              -3.136421
         alpha005 proxy_vwap                 -2.485095         0.722309    -2.244152              -2.038940
         alpha001 baseline_alpha001          -2.929179         0.693653    -2.708545              -2.566692
         alpha036 proxy_vwap                 -1.850211         0.603941    -1.296817              -1.394467
         alpha057 proxy_vwap                 -2.669779         0.515791    -2.615965              -2.835442
         alpha089 snapshot_metadata_risk     -4.485362         0.514243    -3.890809              -3.361547
         alpha090 snapshot_metadata_risk     -4.449852         0.386491    -4.236633              -3.697325
         alpha041 proxy_vwap                 -2.309819         0.340569    -2.390417              -2.044365
nifty500 alpha001 baseline_alpha001          -2.577955         0.170666    -2.431919              -2.525165

## Proxy And Snapshot Risk

In [18]:
display(robustness["proxy_sensitivity"].sort_values(["proxy_dependent", "sharpe_range"], ascending=[False, False]).head(80))
display(robustness["industry_snapshot_risk"].sort_values("snapshot_minus_identity_sharpe", ascending=False).head(40))

,panel,alpha_id,robustness_lane,variant,proxy_check_status,median_signal_rank_corr_vs_hlc3,mean_signal_rank_corr_vs_hlc3,active_sharpe_20bps,active_sharpe_delta_vs_hlc3,reason,min_median_corr,sharpe_range,sharpe_sign_flips,proxy_dependent
28,expanded,alpha057,proxy_vwap,hlc3,ok,1.000000,1.000000,1.027906,0.000000,NaN,-0.004154,0.220856,False,True
29,expanded,alpha057,proxy_vwap,close,ok,-0.004154,-0.003794,1.248762,0.220856,NaN,-0.004154,0.220856,False,True
30,expanded,alpha057,proxy_vwap,ohlc4,ok,0.935532,0.931051,1.127295,0.099389,NaN,-0.004154,0.220856,False,True
31,expanded,alpha057,proxy_vwap,hl2c4,ok,1.000000,1.000000,1.027922,0.000016,NaN,-0.004154,0.220856,False,True
24,expanded,alpha005,proxy_vwap,hlc3,ok,1.000000,1.000000,1.043808,0.000000,NaN,0.545185,0.070278,False,True
25,expanded,alpha005,proxy_vwap,close,ok,0.545185,0.535808,1.039844,-0.003964,NaN,0.545185,0.070278,False,True
26,expanded,alpha005,proxy_vwap,ohlc4,ok,0.967443,0.962101,1.110122,0.066314,NaN,0.545185,0.070278,False,True
27,expanded,alpha005,proxy_vwap,hl2c4,ok,0.999577,0.999445,1.048264,0.004456,NaN,0.545185,0.070278,False,True
12,expanded,alpha041,proxy_vwap,hlc3,ok,1.000000,1.000000,1.160041,0.000000,NaN,0.243714,0.049307,False,True
13,expanded,alpha041,proxy_vwap,close,ok,0.999064,0.998030,1.114971,-0.045070,NaN,0.243714,0.049307,False,True


,panel,alpha_id,robustness_lane,mask,signal_transform,strategy,median_signal_rank_corr_snapshot_vs_identity,mean_signal_rank_corr_snapshot_vs_identity,snapshot_active_sharpe_20bps,identity_active_sharpe_20bps,snapshot_minus_identity_sharpe,final_data_risk,note
2,expanded,alpha069,snapshot_metadata_risk,high_vol_top100,style_residual,overlay20,0.780690,0.761749,1.459798,0.907482,0.552316,snapshot_metadata_risk,Formula-level industry neutralization uses cur...
1,expanded,alpha063,snapshot_metadata_risk,high_vol_top100,rank_centered,overlay20,0.813090,0.806877,1.676850,1.410163,0.266687,snapshot_metadata_risk,Formula-level industry neutralization uses cur...
4,expanded,alpha089,snapshot_metadata_risk,high_vol_top100,winsor_zscore,overlay20,0.529491,0.522505,1.314978,1.094106,0.220873,snapshot_metadata_risk,Formula-level industry neutralization uses cur...
3,expanded,alpha067,snapshot_metadata_risk,high_vol_top100,industry_residual,ewm3_overlay20,0.594595,0.567549,1.441791,1.265074,0.176717,snapshot_metadata_risk,Formula-level industry neutralization uses cur...
6,expanded,alpha090,snapshot_metadata_risk,high_vol_top100,industry_residual,overlay20,0.875823,0.871032,1.063512,1.022182,0.041329,snapshot_metadata_risk,Formula-level industry neutralization uses cur...
5,expanded,alpha070,snapshot_metadata_risk,high_vol_top100,industry_residual,ewm3_overlay20,0.669116,0.655120,1.293148,1.284208,0.008940,snapshot_metadata_risk,Formula-level industry neutralization uses cur...
7,expanded,alpha082,snapshot_metadata_risk,high_vol_top100,industry_residual,overlay20,1.000000,0.999978,1.032999,1.032715,0.000284,snapshot_metadata_risk,Formula-level industry neutralization uses cur...
0,expanded,alpha097,snapshot_metadata_risk,high_vol_top100,rank_centered,ewm3_overlay20,0.987879,0.979984,1.292440,1.292440,0.000000,snapshot_metadata_risk,Formula-level industry neutralization uses cur...


## Preprocessing And Scaling Caveats

The current Alpha101 factory is acceptable as a **candidate discovery and robustness triage layer**, but preprocessing is not yet production-clean. The scaling and transform framework is mostly correct: OHLC fields are adjusted with `adj_close / close`, cross-sectional ranks/z-scores are computed by date inside active masks, `scale()` uses row-wise L1 normalization, orientation is causal, and robustness selection is train-only before OOS scoring.

The remaining preprocessing risks are important before capital promotion:

- Several formula implementations replace invalid rolling values with `0` or `1`; this can convert missing warmup data into signal. A stricter NaN-preserving formula pass is needed.
- Liquidity notional currently uses adjusted close times volume; raw close times volume is better for traded rupee volume and capacity checks.
- VWAP remains a proxy, so proxy-VWAP alphas need real VWAP before promotion. Current proxy sensitivity demotes unstable names, but it does not prove proxy names are fully clean.
- Industry metadata is a current snapshot, not point-in-time; industry-neutral alphas remain research-only under `snapshot_metadata_risk`.
- Current constituent universes carry survivorship/PIT-membership risk.
- Add explicit selected-name forward-return, stale-price, warmup-NaN, and invalid-return audits before treating promoted candidates as tradable.

Practical conclusion: **scaling is mostly fine; preprocessing/data-validity hygiene is the next correctness gate.** Rerun promoted exact-OHLCV candidates after a strict NaN and liquidity-notional audit.


## Robustness Final Report

In [19]:
robust_report_path = ARTIFACT_DIR / "alpha101_robustness_final_report.md"
if robust_report_path.exists():
    display(Markdown(robust_report_path.read_text()))
else:
    display(Markdown("Robustness report has not been written yet. Run the robustness cell above."))

# Alpha101 Robustness And Data-Risk Triage Report

This report treats the Alpha101 factory output as discovery only, then reruns top candidates with train-only transform/portfolio selection, active benchmark comparison, cost sensitivity, and proxy/snapshot data-risk checks.

## Candidate Lane Counts
| robustness_lane | rows |
| --- | --- |
| clean_exact_ohlcv | 12 |
| proxy_vwap | 8 |
| snapshot_metadata_risk | 8 |
| baseline_alpha001 | 2 |

## Final Status Counts
| final_status | candidates |
| --- | --- |
| promote_to_deeper_research | 16 |
| snapshot_metadata_risk | 8 |
| proxy_dependent | 3 |
| baseline_comparator | 2 |
| feature_only | 1 |

## Top Robustness Rows
| panel | alpha_id | robustness_lane | input_quality_tier | final_status | median_test_active_sharpe | median_test_active_cagr | positive_test_sharpe_rate | median_test_rank_ic | median_turnover |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| expanded | alpha063 | snapshot_metadata_risk | proxy_vwap+snapshot_industry | snapshot_metadata_risk | 1.744367771963078 | 0.028139071996760134 | 0.75 | 0.03110226344659866 | 0.06581288241984556 |
| expanded | alpha040 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.6697132474056275 | 0.02828268524745059 | 1.0 | 0.03454638977998095 | 0.0661793516299495 |
| expanded | alpha012 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.4485779860062387 | 0.02672855495037796 | 1.0 | 0.020489551516343116 | 0.06696880233674198 |
| expanded | alpha024 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.4419888045917393 | 0.02748926501232296 | 1.0 | 0.03932020819847223 | 0.0629369109715853 |
| expanded | alpha044 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.364652727586496 | 0.022774025977340684 | 1.0 | 0.041259868320261454 | 0.06788229508642284 |
| expanded | alpha069 | snapshot_metadata_risk | proxy_vwap+snapshot_industry | snapshot_metadata_risk | 1.349113986059259 | 0.022691819001740776 | 1.0 | 0.008950670182696501 | 0.06667744527716941 |
| expanded | alpha018 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.3415998145121595 | 0.024728907268033118 | 1.0 | 0.022731160144445175 | 0.0658981934542523 |
| expanded | alpha023 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.2689376717973546 | 0.024178799774613347 | 1.0 | 0.029527779036525197 | 0.06760257324210854 |
| expanded | alpha051 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.2130137422953249 | 0.023336582997883548 | 1.0 | 0.016317717136332673 | 0.06809226796313954 |
| expanded | alpha049 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.2054578292051223 | 0.023192778721148888 | 0.75 | 0.017777233199822058 | 0.06809277690470616 |
| expanded | alpha067 | snapshot_metadata_risk | proxy_vwap+snapshot_industry | snapshot_metadata_risk | 1.196730901480091 | 0.02040533904137032 | 1.0 | 0.013540971590154256 | 0.0673069847450729 |
| expanded | alpha016 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.183597401097284 | 0.02104247700601436 | 1.0 | 0.028470568916695627 | 0.06678959291544936 |
| expanded | alpha034 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.087923111840096 | 0.01834400969626071 | 1.0 | 0.027965210152839926 | 0.06676635779171633 |
| expanded | alpha094 | proxy_vwap | proxy_vwap | promote_to_deeper_research | 1.0669455871956994 | 0.020768259313892035 | 1.0 | 0.011979742345960307 | 0.0659092045222993 |
| expanded | alpha025 | proxy_vwap | proxy_vwap | promote_to_deeper_research | 1.0482335763780388 | 0.02001227591914989 | 1.0 | 0.03452867770948616 | 0.06619664213684019 |
| expanded | alpha022 | clean_exact_ohlcv | exact_ohlcv | promote_to_deeper_research | 1.0040157786323565 | 0.01607636625011588 | 1.0 | 0.020154424032491146 | 0.06920631803255028 |
| expanded | alpha070 | snapshot_metadata_risk | proxy_vwap+snapshot_industry | snapshot_metadata_risk | 0.93237266445517 | 0.01648963882305088 | 1.0 | 0.018765056954752557 | 0.06771065047095633 |
| expanded | alpha050 | proxy_vwap | proxy_vwap | promote_to_deeper_research | 0.9012238741380603 | 0.015515649927456243 | 0.75 | 0.018350115726394157 | 0.06759000520640186 |
| expanded | alpha082 | snapshot_metadata_risk | snapshot_industry | snapshot_metadata_risk | 0.8893334993778996 | 0.01686882680552737 | 1.0 | 0.010088171490961941 | 0.06675031886885636 |
| expanded | alpha097 | snapshot_metadata_risk | proxy_vwap+snapshot_industry | snapshot_metadata_risk | 0.8804017493541336 | 0.013326248372700134 | 1.0 |  | 0.05824847534427714 |
| expanded | alpha027 | proxy_vwap | proxy_vwap | promote_to_deeper_research | 0.7466265248459774 | 0.01231934002777857 | 0.75 | 0.019638943539851256 | 0.06647161276135409 |
| expanded | alpha005 | proxy_vwap | proxy_vwap | proxy_dependent | 0.7223093399150153 | 0.013401326690783422 | 1.0 | 0.02065734553695906 | 0.06778418577601919 |
| expanded | alpha001 | baseline_alpha001 | exact_ohlcv | baseline_comparator | 0.6936531205501564 | 0.011994270700546927 | 1.0 | 0.02255003200152462 | 0.0670979629419882 |
| expanded | alpha036 | proxy_vwap | proxy_vwap | promote_to_deeper_research | 0.603941310200634 | 0.009885836014942417 | 0.75 | 0.01732359422563584 | 0.06373655447269672 |
| expanded | alpha057 | proxy_vwap | proxy_vwap | proxy_dependent | 0.5157909665914726 | 0.009320556372320099 | 0.75 | 0.01717771533404909 | 0.0660089609688813 |

## Proxy Sensitivity Summary
| panel | alpha_id | proxy_dependent | min_median_corr | sharpe_range |
| --- | --- | --- | --- | --- |
| expanded | alpha057 | True | -0.004154150214165792 | 0.2208557427746849 |
| expanded | alpha005 | True | 0.5451850607268014 | 0.07027808668832725 |
| expanded | alpha041 | True | 0.24371429600115213 | 0.04930709299451452 |
| expanded | alpha050 | False | 0.7438725718141465 | 0.11300316406995203 |
| expanded | alpha027 | False | 0.7204131439939278 | 0.09159361998159721 |
| expanded | alpha070 | False | 0.889409618599876 | 0.09044850346289879 |
| expanded | alpha069 | False | 0.946894689468947 | 0.08396984979111721 |
| expanded | alpha089 | False | 0.978458359405192 | 0.061060346639744134 |
| expanded | alpha094 | False | 0.9511890410730345 | 0.0263190163455016 |
| expanded | alpha036 | False | 0.9869819945156344 | 0.018328647304961088 |
| expanded | alpha067 | False | 0.9763811669470737 | 0.01229741528012629 |
| expanded | alpha063 | False | 0.9994928880643167 | 0.007306623067637963 |
| expanded | alpha025 | False | 0.9998919891989199 | 0.0011372166412788598 |
| expanded | alpha097 | False | 1.0 | 0.0 |

## Industry Snapshot Risk Summary
| panel | alpha_id | median_signal_rank_corr_snapshot_vs_identity | snapshot_minus_identity_sharpe | final_data_risk |
| --- | --- | --- | --- | --- |
| expanded | alpha097 | 0.9878787878787879 | 0.0 | snapshot_metadata_risk |
| expanded | alpha063 | 0.8130900705371676 | 0.26668691882015216 | snapshot_metadata_risk |
| expanded | alpha069 | 0.7806900690069007 | 0.5523163407978073 | snapshot_metadata_risk |
| expanded | alpha067 | 0.594594650204343 | 0.17671748512686025 | snapshot_metadata_risk |
| expanded | alpha089 | 0.529491279556744 | 0.22087276066584716 | snapshot_metadata_risk |
| expanded | alpha070 | 0.6691156462585034 | 0.008940131892981151 | snapshot_metadata_risk |
| expanded | alpha090 | 0.8758228372207454 | 0.041329414269857034 | snapshot_metadata_risk |
| expanded | alpha082 | 1.0 | 0.0002835890045185252 | snapshot_metadata_risk |

## Interpretation Rules
- `promote_to_deeper_research` means the candidate survived walk-forward active return tests after train-only selection.
- `feature_only` means IC survived better than portfolio expression.
- `proxy_dependent` means VWAP proxy choice materially changes signal rankings or active Sharpe.
- `snapshot_metadata_risk` remains research-only until point-in-time industry metadata exists.
- `baseline_comparator` is included for Alpha#1 context only.

## Batch 2 Clean Near-Miss Robustness

This reruns only the 20 clean exact-OHLCV near-miss candidates from discovery. It uses the same train-only walk-forward selection, active benchmark comparison, universe sensitivity, and cost grid as Batch 1, while excluding proxy, snapshot-industry, decayed, and discarded alphas.

In [20]:
batch2 = run_alpha101_robustness_batch2(
    refresh=RUN_ALPHA101_ROBUSTNESS_BATCH2_REFRESH,
    progress=True,
)
{k: v.shape for k, v in batch2.items()}

{'candidate_lanes': (20, 20),
 'walk_forward': (80, 30),
 'cost_sensitivity': (320, 30),
 'universe_sensitivity': (320, 30),
 'validation': (6, 3),
 'shortlist': (20, 20),
 'combined_shortlist': (50, 21)}

## Batch 2 Validation

In [21]:
display(batch2["validation"])
assert batch2["validation"]["passed"].all(), "Batch 2 robustness validation failed"

,check,passed,detail
0,batch2_exact_20_alphas,True,Expected 20 fixed clean near-miss alphas; foun...
1,batch2_only_exact_ohlcv,True,"Batch 2 excludes proxy, snapshot, decayed, and..."
2,batch2_train_selected_only,True,Walk-forward rows select transforms/portfolios...
3,batch2_cost_grid_present,True,Cost sensitivity exists at 10/20/35/50 bps.
4,batch2_universe_masks_present,True,"Universe sensitivity includes all eligible, hi..."
5,batch2_no_batch1_overwrite,True,Batch 1 artifact files are still present.


## Batch 2 Shortlist

Batch 2 uses the same final status rules as Batch 1: `promote_to_deeper_research`, `feature_only`, or `discard`. These are still research classifications, not capital allocation approvals.

In [22]:
batch2_cols = [
    "panel", "alpha_id", "robustness_lane", "input_quality_tier", "final_status",
    "median_test_active_sharpe", "median_test_active_cagr", "positive_test_sharpe_rate",
    "median_test_rank_ic", "median_turnover", "worst_test_drawdown",
    "best_mask", "best_signal_transform", "best_strategy",
]
display(batch2["shortlist"].sort_values("median_test_active_sharpe", ascending=False)[batch2_cols])
display(batch2["shortlist"].groupby("final_status", dropna=False).size().rename("count").reset_index())

,panel,alpha_id,robustness_lane,input_quality_tier,final_status,median_test_active_sharpe,median_test_active_cagr,positive_test_sharpe_rate,median_test_rank_ic,median_turnover,worst_test_drawdown,best_mask,best_signal_transform,best_strategy
3,expanded,alpha026,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,1.505659,0.026694,1.00,0.030148,0.067531,-0.022309,high_vol_top100,winsor_zscore,overlay20
4,expanded,alpha033,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,1.165859,0.021484,1.00,0.031767,0.066210,-0.026420,high_vol_top100,ewm3,ewm3_overlay20
5,expanded,alpha013,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,1.137544,0.019149,1.00,0.022281,0.066848,-0.021881,high_vol_top100,ewm3,overlay20
6,expanded,alpha015,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,1.086061,0.018113,0.75,0.022677,0.068160,-0.022338,high_vol_top100,rank_centered,overlay20
7,expanded,alpha003,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,0.941490,0.015570,0.75,0.011634,0.067164,-0.028000,high_vol_top100,rank_centered,overlay20
8,expanded,alpha045,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,0.904456,0.014872,1.00,0.014302,0.067072,-0.017241,high_vol_top100,ewm3,ewm3_overlay20
9,expanded,alpha038,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,0.889312,0.017319,1.00,0.025777,0.067929,-0.027603,high_vol_top100,winsor_zscore,ewm3_overlay20
10,expanded,alpha068,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,0.881240,0.014694,0.75,0.015941,0.067140,-0.021946,high_vol_top100,liquidity_scaled,overlay20
11,expanded,alpha088,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,0.720715,0.013810,0.75,0.031329,0.070673,-0.024326,high_vol_top100,rank_centered,overlay20
12,expanded,alpha010,clean_near_miss_batch2,exact_ohlcv,promote_to_deeper_research,0.700723,0.012554,1.00,0.019535,0.065303,-0.021879,high_vol_top100,ewm3,ewm5_overlay20


,final_status,count
0,feature_only,3
1,promote_to_deeper_research,17


## Combined Exact-OHLCV Promotions

This table keeps Batch 1 and Batch 2 separate, then filters to promoted exact-OHLCV candidates so the clean research lane is visible without proxy or snapshot metadata risk mixed in.

In [23]:
combined_promoted_exact = batch2["combined_shortlist"].loc[
    batch2["combined_shortlist"]["final_status"].eq("promote_to_deeper_research")
    & batch2["combined_shortlist"].get("input_quality_tier", pd.Series(index=batch2["combined_shortlist"].index, dtype=object)).eq("exact_ohlcv")
].sort_values(["batch", "median_test_active_sharpe"], ascending=[True, False])

combined_cols = [
    "batch", "panel", "alpha_id", "robustness_lane", "final_status",
    "median_test_active_sharpe", "median_test_active_cagr", "positive_test_sharpe_rate",
    "median_test_rank_ic", "median_turnover", "best_mask", "best_signal_transform", "best_strategy",
]
display(combined_promoted_exact[combined_cols])
display(combined_promoted_exact.groupby("batch", dropna=False).size().rename("promoted_exact_ohlcv_count").reset_index())

,batch,panel,alpha_id,robustness_lane,final_status,median_test_active_sharpe,median_test_active_cagr,positive_test_sharpe_rate,median_test_rank_ic,median_turnover,best_mask,best_signal_transform,best_strategy
6,batch1,expanded,alpha040,clean_exact_ohlcv,promote_to_deeper_research,1.669713,0.028283,1.00,0.034546,0.066179,high_vol_top100,winsor_zscore,overlay20
8,batch1,expanded,alpha012,clean_exact_ohlcv,promote_to_deeper_research,1.448578,0.026729,1.00,0.020490,0.066969,high_vol_top100,ewm3,ewm3_overlay20
9,batch1,expanded,alpha024,clean_exact_ohlcv,promote_to_deeper_research,1.441989,0.027489,1.00,0.039320,0.062937,high_vol_top100,rank_centered,ewm5_overlay20
10,batch1,expanded,alpha044,clean_exact_ohlcv,promote_to_deeper_research,1.364653,0.022774,1.00,0.041260,0.067882,high_vol_top100,ewm3,overlay20
11,batch1,expanded,alpha018,clean_exact_ohlcv,promote_to_deeper_research,1.341600,0.024729,1.00,0.022731,0.065898,high_vol_top100,ewm3,ewm3_overlay20
12,batch1,expanded,alpha023,clean_exact_ohlcv,promote_to_deeper_research,1.268938,0.024179,1.00,0.029528,0.067603,high_vol_top100,winsor_zscore,ewm5_overlay20
13,batch1,expanded,alpha051,clean_exact_ohlcv,promote_to_deeper_research,1.213014,0.023337,1.00,0.016318,0.068092,high_vol_top100,winsor_zscore,ewm5_overlay20
14,batch1,expanded,alpha049,clean_exact_ohlcv,promote_to_deeper_research,1.205458,0.023193,0.75,0.017777,0.068093,high_vol_top100,winsor_zscore,ewm5_overlay20
15,batch1,expanded,alpha016,clean_exact_ohlcv,promote_to_deeper_research,1.183597,0.021042,1.00,0.028471,0.066790,high_vol_top100,ewm3,overlay20
18,batch1,expanded,alpha034,clean_exact_ohlcv,promote_to_deeper_research,1.087923,0.018344,1.00,0.027965,0.066766,high_vol_top100,ewm3,ewm3_overlay20


,batch,promoted_exact_ohlcv_count
0,batch1,11
1,batch2,17


## Batch 2 Final Report

In [24]:
batch2_report_path = ARTIFACT_DIR / "alpha101_robustness_batch2_final_report.md"
if batch2_report_path.exists():
    display(Markdown(batch2_report_path.read_text()))
else:
    display(Markdown("Batch 2 robustness report has not been written yet. Run the Batch 2 cell above."))

# Alpha101 Robustness Batch 2 Clean Near-Miss Report

Batch 2 reruns only exact-OHLCV discovery candidates that missed the first top-12 robustness cutoff. It excludes proxy, snapshot-industry, decayed, and discarded alphas.

## Batch 2 Lane Counts
| robustness_lane | rows |
| --- | --- |
| clean_near_miss_batch2 | 20 |

## Batch 2 Final Status Counts
| final_status | candidates |
| --- | --- |
| promote_to_deeper_research | 17 |
| feature_only | 3 |

## Batch 2 Results
| panel | alpha_id | robustness_lane | final_status | median_test_active_sharpe | median_test_active_cagr | positive_test_sharpe_rate | median_test_rank_ic | median_turnover |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| expanded | alpha026 | clean_near_miss_batch2 | promote_to_deeper_research | 1.5056593269482264 | 0.0266942662450238 | 1.0 | 0.030147566165233584 | 0.06753089428264267 |
| expanded | alpha033 | clean_near_miss_batch2 | promote_to_deeper_research | 1.1658593546969578 | 0.02148400673163875 | 1.0 | 0.031767072938202724 | 0.06621000779336106 |
| expanded | alpha013 | clean_near_miss_batch2 | promote_to_deeper_research | 1.1375441286741546 | 0.019149472735586204 | 1.0 | 0.02228091389794271 | 0.0668478946472313 |
| expanded | alpha015 | clean_near_miss_batch2 | promote_to_deeper_research | 1.0860612424957163 | 0.0181127144380685 | 0.75 | 0.022677057737691082 | 0.06815993127619893 |
| expanded | alpha003 | clean_near_miss_batch2 | promote_to_deeper_research | 0.941490319759053 | 0.015570036284664357 | 0.75 | 0.011633950440654628 | 0.06716403560812792 |
| expanded | alpha045 | clean_near_miss_batch2 | promote_to_deeper_research | 0.9044557720529969 | 0.014872031574506894 | 1.0 | 0.014302130320542908 | 0.06707187831768127 |
| expanded | alpha038 | clean_near_miss_batch2 | promote_to_deeper_research | 0.8893115704379876 | 0.01731928296882268 | 1.0 | 0.025777390568946665 | 0.06792932144747515 |
| expanded | alpha068 | clean_near_miss_batch2 | promote_to_deeper_research | 0.8812400978511057 | 0.01469448594738132 | 0.75 | 0.015940562440471342 | 0.06714035912771607 |
| expanded | alpha088 | clean_near_miss_batch2 | promote_to_deeper_research | 0.7207148776779675 | 0.013810418877810648 | 0.75 | 0.03132907881009597 | 0.07067286071059559 |
| expanded | alpha010 | clean_near_miss_batch2 | promote_to_deeper_research | 0.7007225103230329 | 0.01255402512170245 | 1.0 | 0.01953487907899797 | 0.06530278266147962 |
| expanded | alpha037 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6774158666539971 | 0.012192277023574838 | 0.75 | 0.015874923244092152 | 0.06246567799370053 |
| expanded | alpha017 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6767918962414987 | 0.012479055747429135 | 1.0 | 0.014811532567165681 | 0.06804389015919993 |
| expanded | alpha004 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6641590612808167 | 0.012151505378471428 | 1.0 | 0.022507086687227187 | 0.06805716376511932 |
| expanded | alpha006 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6596611631891723 | 0.01059154164071141 | 0.75 | 0.013471073479858087 | 0.06686002593094263 |
| expanded | alpha007 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6334636503925335 | 0.01249734107256062 | 0.75 | 0.01898251824056771 | 0.06982657812962627 |
| expanded | alpha055 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6159992294929755 | 0.01012915592436281 | 0.75 | 0.02041191734115292 | 0.06784748829394867 |
| expanded | alpha002 | clean_near_miss_batch2 | feature_only | 0.6114709064957689 | 0.010342026415863348 | 0.5 | 0.01850213216066612 | 0.06850498933074793 |
| expanded | alpha009 | clean_near_miss_batch2 | promote_to_deeper_research | 0.5429917978311637 | 0.009590488156750787 | 1.0 | 0.024941180977136344 | 0.06562062173141862 |
| expanded | alpha028 | clean_near_miss_batch2 | feature_only | 0.3908696219422299 | 0.007260253901404223 | 0.5 | 0.012905929998896662 | 0.06761228216798185 |
| expanded | alpha008 | clean_near_miss_batch2 | feature_only | 0.37692261363315127 | 0.006369399206878179 | 1.0 | 0.01961470784682228 | 0.06868266007973482 |

## Combined Promoted Exact-OHLCV Candidates
| batch | panel | alpha_id | robustness_lane | final_status | median_test_active_sharpe | median_test_active_cagr | positive_test_sharpe_rate | median_test_rank_ic | median_turnover |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| batch1 | expanded | alpha040 | clean_exact_ohlcv | promote_to_deeper_research | 1.6697132474056275 | 0.0282826852474505 | 1.0 | 0.0345463897799809 | 0.0661793516299495 |
| batch1 | expanded | alpha012 | clean_exact_ohlcv | promote_to_deeper_research | 1.4485779860062389 | 0.0267285549503779 | 1.0 | 0.0204895515163431 | 0.0669688023367419 |
| batch1 | expanded | alpha024 | clean_exact_ohlcv | promote_to_deeper_research | 1.4419888045917393 | 0.0274892650123229 | 1.0 | 0.0393202081984722 | 0.0629369109715853 |
| batch1 | expanded | alpha044 | clean_exact_ohlcv | promote_to_deeper_research | 1.364652727586496 | 0.0227740259773406 | 1.0 | 0.0412598683202614 | 0.0678822950864228 |
| batch1 | expanded | alpha018 | clean_exact_ohlcv | promote_to_deeper_research | 1.3415998145121597 | 0.0247289072680331 | 1.0 | 0.0227311601444451 | 0.0658981934542523 |
| batch1 | expanded | alpha023 | clean_exact_ohlcv | promote_to_deeper_research | 1.2689376717973546 | 0.0241787997746133 | 1.0 | 0.0295277790365251 | 0.0676025732421085 |
| batch1 | expanded | alpha051 | clean_exact_ohlcv | promote_to_deeper_research | 1.2130137422953249 | 0.0233365829978835 | 1.0 | 0.0163177171363326 | 0.0680922679631395 |
| batch1 | expanded | alpha049 | clean_exact_ohlcv | promote_to_deeper_research | 1.2054578292051223 | 0.0231927787211488 | 0.75 | 0.017777233199822 | 0.0680927769047061 |
| batch1 | expanded | alpha016 | clean_exact_ohlcv | promote_to_deeper_research | 1.183597401097284 | 0.0210424770060143 | 1.0 | 0.0284705689166956 | 0.0667895929154493 |
| batch1 | expanded | alpha034 | clean_exact_ohlcv | promote_to_deeper_research | 1.087923111840096 | 0.0183440096962607 | 1.0 | 0.0279652101528399 | 0.0667663577917163 |
| batch1 | expanded | alpha022 | clean_exact_ohlcv | promote_to_deeper_research | 1.0040157786323565 | 0.0160763662501158 | 1.0 | 0.0201544240324911 | 0.0692063180325502 |
| batch2 | expanded | alpha026 | clean_near_miss_batch2 | promote_to_deeper_research | 1.5056593269482264 | 0.0266942662450238 | 1.0 | 0.030147566165233584 | 0.06753089428264267 |
| batch2 | expanded | alpha033 | clean_near_miss_batch2 | promote_to_deeper_research | 1.1658593546969578 | 0.02148400673163875 | 1.0 | 0.031767072938202724 | 0.06621000779336106 |
| batch2 | expanded | alpha013 | clean_near_miss_batch2 | promote_to_deeper_research | 1.1375441286741546 | 0.019149472735586204 | 1.0 | 0.02228091389794271 | 0.0668478946472313 |
| batch2 | expanded | alpha015 | clean_near_miss_batch2 | promote_to_deeper_research | 1.0860612424957163 | 0.0181127144380685 | 0.75 | 0.022677057737691082 | 0.06815993127619893 |
| batch2 | expanded | alpha003 | clean_near_miss_batch2 | promote_to_deeper_research | 0.941490319759053 | 0.015570036284664357 | 0.75 | 0.011633950440654628 | 0.06716403560812792 |
| batch2 | expanded | alpha045 | clean_near_miss_batch2 | promote_to_deeper_research | 0.9044557720529969 | 0.014872031574506894 | 1.0 | 0.014302130320542908 | 0.06707187831768127 |
| batch2 | expanded | alpha038 | clean_near_miss_batch2 | promote_to_deeper_research | 0.8893115704379876 | 0.01731928296882268 | 1.0 | 0.025777390568946665 | 0.06792932144747515 |
| batch2 | expanded | alpha068 | clean_near_miss_batch2 | promote_to_deeper_research | 0.8812400978511057 | 0.01469448594738132 | 0.75 | 0.015940562440471342 | 0.06714035912771607 |
| batch2 | expanded | alpha088 | clean_near_miss_batch2 | promote_to_deeper_research | 0.7207148776779675 | 0.013810418877810648 | 0.75 | 0.03132907881009597 | 0.07067286071059559 |
| batch2 | expanded | alpha010 | clean_near_miss_batch2 | promote_to_deeper_research | 0.7007225103230329 | 0.01255402512170245 | 1.0 | 0.01953487907899797 | 0.06530278266147962 |
| batch2 | expanded | alpha037 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6774158666539971 | 0.012192277023574838 | 0.75 | 0.015874923244092152 | 0.06246567799370053 |
| batch2 | expanded | alpha017 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6767918962414987 | 0.012479055747429135 | 1.0 | 0.014811532567165681 | 0.06804389015919993 |
| batch2 | expanded | alpha004 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6641590612808167 | 0.012151505378471428 | 1.0 | 0.022507086687227187 | 0.06805716376511932 |
| batch2 | expanded | alpha006 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6596611631891723 | 0.01059154164071141 | 0.75 | 0.013471073479858087 | 0.06686002593094263 |
| batch2 | expanded | alpha007 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6334636503925335 | 0.01249734107256062 | 0.75 | 0.01898251824056771 | 0.06982657812962627 |
| batch2 | expanded | alpha055 | clean_near_miss_batch2 | promote_to_deeper_research | 0.6159992294929755 | 0.01012915592436281 | 0.75 | 0.02041191734115292 | 0.06784748829394867 |
| batch2 | expanded | alpha009 | clean_near_miss_batch2 | promote_to_deeper_research | 0.5429917978311637 | 0.009590488156750787 | 1.0 | 0.024941180977136344 | 0.06562062173141862 |

## Interpretation Rules
- `promote_to_deeper_research` uses the same Batch 1 robustness thresholds.
- `feature_only` means IC survived better than portfolio expression.
- `discard` means the alpha failed the Batch 2 walk-forward robustness hurdle.
- Batch 2 remains a research expansion, not a capital-promotion step.

## Final Report

In [25]:
report_path = ARTIFACT_DIR / "alpha101_final_report.md"
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    display(Markdown("Final report has not been written yet. Run the factory cell above."))

# Alpha101 Research Factory Report

This report evaluates the Kakushadze 101 Formulaic Alphas on the cached NIFTY500 and expanded India equity universes.

## Classification Counts
| classification | alpha_panel_rows |
| --- | --- |
| discard | 93 |
| candidate | 84 |
| feature_only | 19 |
| decayed | 4 |
| untestable | 1 |

## Input Quality Counts
| input_quality_tier | alphas |
| --- | --- |
| exact_ohlcv | 51 |
| proxy_vwap | 31 |
| proxy_vwap+snapshot_industry | 14 |
| snapshot_industry | 4 |
| missing_cap | 1 |

## Formula Failures / Untestable
| panel | alpha_id | reason |
| --- | --- | --- |
| nifty500 | alpha056 | market cap is unavailable in current cache |
| expanded | alpha056 | market cap is unavailable in current cache |

## Top 25 Research Scores
| panel | alpha_id | family | input_quality_tier | classification | best_5d_ic | best_20bps_active_sharpe | best_signal_transform | best_strategy | best_mask | research_score |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| expanded | alpha097 | industry_neutral_cross_section | proxy_vwap+snapshot_industry | candidate | 0.2092736409724257 | 1.292440255029393 | rank_centered | ewm3_overlay20 | high_vol_top100 | 2.476420254098931 |
| expanded | alpha024 | price_reversal | exact_ohlcv | candidate | 0.0493286444718435 | 1.8305163296173488 | rank_centered | ewm5_overlay20 | high_vol_top100 | 2.2130096497235847 |
| expanded | alpha023 | price_reversal | exact_ohlcv | candidate | 0.0433553243864415 | 1.830744627306076 | winsor_zscore | ewm5_overlay20 | high_vol_top100 | 2.1772361532572115 |
| expanded | alpha063 | industry_neutral_cross_section | proxy_vwap+snapshot_industry | candidate | 0.0450201357306716 | 1.6768501007803094 | rank_centered | overlay20 | high_vol_top100 | 2.035332904602581 |
| expanded | alpha034 | price_reversal | exact_ohlcv | candidate | 0.0363145266040637 | 1.6832472606867983 | ewm3 | ewm3_overlay20 | high_vol_top100 | 2.003309651409447 |
| expanded | alpha044 | volume_liquidity | exact_ohlcv | candidate | 0.0565560157247093 | 1.5773518963646616 | ewm3 | overlay20 | high_vol_top100 | 2.0005381633268082 |
| expanded | alpha016 | volume_liquidity | exact_ohlcv | candidate | 0.0590261894827582 | 1.5388556394263375 | ewm3 | overlay20 | high_vol_top100 | 1.9747144651563746 |
| nifty500 | alpha024 | price_reversal | exact_ohlcv | candidate | 0.0429589389344959 | 1.5589327667190012 | winsor_zscore | ewm3_overlay20 | seasoned_2y | 1.9249997928693643 |
| expanded | alpha040 | volume_liquidity | exact_ohlcv | candidate | 0.0407693666046218 | 1.5835098339480758 | winsor_zscore | overlay20 | high_vol_top100 | 1.9187824883133942 |
| expanded | alpha049 | price_reversal | exact_ohlcv | candidate | 0.0300630063238555 | 1.5532610274051717 | winsor_zscore | ewm5_overlay20 | high_vol_top100 | 1.8246866860214825 |
| expanded | alpha051 | price_reversal | exact_ohlcv | candidate | 0.0285317074362419 | 1.5473759988570803 | winsor_zscore | ewm5_overlay20 | high_vol_top100 | 1.810969584230751 |
| expanded | alpha025 | volume_liquidity | proxy_vwap | candidate | 0.0415065523615567 | 1.462949165574519 | ewm3 | ewm3_overlay20 | high_vol_top100 | 1.8082569512085276 |
| expanded | alpha018 | volume_liquidity | exact_ohlcv | candidate | 0.0331925812819151 | 1.501092166240621 | ewm3 | ewm3_overlay20 | high_vol_top100 | 1.7948610625889108 |
| expanded | alpha022 | volume_liquidity | exact_ohlcv | candidate | 0.0292880750844989 | 1.4715889504021389 | ewm3 | ewm3_overlay20 | high_vol_top100 | 1.7469852281757385 |
| expanded | alpha069 | industry_neutral_cross_section | proxy_vwap+snapshot_industry | candidate | 0.0277408960712405 | 1.4597984309358725 | style_residual | overlay20 | high_vol_top100 | 1.7253376907595321 |
| expanded | alpha012 | price_reversal | exact_ohlcv | candidate | 0.0340859786177538 | 1.4187610788804903 | ewm3 | ewm3_overlay20 | high_vol_top100 | 1.7230325745606812 |
| expanded | alpha026 | volume_liquidity | exact_ohlcv | candidate | 0.0436906028991482 | 1.345286884488185 | winsor_zscore | overlay20 | high_vol_top100 | 1.7026542768625408 |
| expanded | alpha013 | volume_liquidity | exact_ohlcv | candidate | 0.048193909697113 | 1.3240603700569842 | ewm3 | overlay20 | high_vol_top100 | 1.6991978620650063 |
| expanded | alpha067 | industry_neutral_cross_section | proxy_vwap+snapshot_industry | candidate | 0.0262844415696481 | 1.4417911959766014 | industry_residual | ewm3_overlay20 | high_vol_top100 | 1.6984453655480676 |
| expanded | alpha033 | price_reversal | exact_ohlcv | candidate | 0.0331424373836405 | 1.3870057165244445 | ewm3 | ewm3_overlay20 | high_vol_top100 | 1.6863466547309716 |
| expanded | alpha075 | volume_liquidity | proxy_vwap | discard | 0.0195601000266463 | 1.440463879434338 | ewm3 | ewm3_overlay20 | high_vol_top100 | 1.6658132306442721 |
| expanded | alpha061 | volume_liquidity | proxy_vwap | discard | 0.011023710262694 | 1.4539329474147014 | ewm3 | ewm3_overlay20 | high_vol_top100 | 1.6285313369470336 |
| expanded | alpha015 | volume_liquidity | exact_ohlcv | candidate | 0.0297105384988163 | 1.3305663486372294 | rank_centered | overlay20 | high_vol_top100 | 1.6077829920596731 |
| expanded | alpha089 | industry_neutral_cross_section | proxy_vwap+snapshot_industry | candidate | 0.0321453961122648 | 1.314978277480208 | winsor_zscore | overlay20 | high_vol_top100 | 1.6044042911810406 |
| expanded | alpha081 | volume_liquidity | proxy_vwap | discard | 0.0177043977439067 | 1.3816063885405268 | ewm3 | ewm3_overlay20 | high_vol_top100 | 1.5978041716105849 |

## Limitations
- VWAP is proxied by typical price `(high + low + close) / 3`.
- Industry neutralization uses current snapshot industry metadata.
- Parent constituent histories are current snapshots, so expanded-universe results carry PIT/survivorship risk.
- Transform winners are research candidates; transform selection is reported separately to control data-mining risk.

## Artifact Manifest

In [26]:
artifact_names = [
    "alpha101_formula_registry.csv",
    "alpha101_input_quality_report.csv",
    "alpha101_operator_validation.csv",
    "alpha101_formula_validation.csv",
    "alpha101_family_classification.csv",
    "alpha101_transform_compatibility.csv",
    "alpha101_metric_panel.csv",
    "alpha101_transform_grid.csv",
    "alpha101_decay_report.csv",
    "alpha101_portfolio_report.csv",
    "alpha101_leaderboard.csv",
    "alpha101_candidate_shortlist.csv",
    "alpha101_final_report.md",
    "alpha101_robustness_final_report.md",
    "alpha101_robustness_shortlist.csv",
    "alpha101_robustness_validation.csv",
    "alpha101_industry_snapshot_risk_report.csv",
    "alpha101_proxy_sensitivity_report.csv",
    "alpha101_robustness_universe_sensitivity.csv",
    "alpha101_robustness_cost_sensitivity.csv",
    "alpha101_robustness_walk_forward.csv",
    "alpha101_robustness_candidate_lanes.csv",
    "alpha101_robustness_batch2_final_report.md",
    "alpha101_robustness_batch2_shortlist.csv",
    "alpha101_robustness_batch2_validation.csv",
    "alpha101_robustness_batch2_universe_sensitivity.csv",
    "alpha101_robustness_batch2_cost_sensitivity.csv",
    "alpha101_robustness_batch2_walk_forward.csv",
    "alpha101_robustness_batch2_candidate_lanes.csv",
    "alpha101_robustness_combined_shortlist.csv",
]
manifest = pd.DataFrame({
    "artifact": artifact_names,
    "path": [str(ARTIFACT_DIR / name) for name in artifact_names],
    "exists": [(ARTIFACT_DIR / name).exists() for name in artifact_names],
})
display(manifest)

,artifact,path,exists
0,alpha101_formula_registry.csv,research/artifacts/alpha101_research_factory/a...,True
1,alpha101_input_quality_report.csv,research/artifacts/alpha101_research_factory/a...,True
2,alpha101_operator_validation.csv,research/artifacts/alpha101_research_factory/a...,True
3,alpha101_formula_validation.csv,research/artifacts/alpha101_research_factory/a...,True
4,alpha101_family_classification.csv,research/artifacts/alpha101_research_factory/a...,True
5,alpha101_transform_compatibility.csv,research/artifacts/alpha101_research_factory/a...,True
6,alpha101_metric_panel.csv,research/artifacts/alpha101_research_factory/a...,True
7,alpha101_transform_grid.csv,research/artifacts/alpha101_research_factory/a...,True
8,alpha101_decay_report.csv,research/artifacts/alpha101_research_factory/a...,True
9,alpha101_portfolio_report.csv,research/artifacts/alpha101_research_factory/a...,True
